# Zimbabwe Bayesian Migration Risk Model - Colab Notebook
## Complete pipeline: Data → Model → Results

## Cell 1: Install Dependencies

In [ ]:
!pip install pymc arviz geopandas pandas numpy matplotlib scikit-learn streamlit streamlit-folium folium -q
print("✓ All dependencies installed!")

## Cell 2: Generate Data Directly

In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

print("="*70)
print("GENERATING ZIMBABWE MIGRATION DATASET")
print("="*70)

# Base districts
districts_list = ['Harare', 'Bulawayo', 'Manicaland', 'Masvingo', 'Midlands',
                  'Mashonaland Central', 'Mashonaland East', 'Mashonaland West',
                  'Matabeleland North', 'Matabeleland South', 'Chitungwiza',
                  'Norton', 'Bindura', 'Gweru', 'Kadoma']

provinces = {
    'Harare': 'Harare',
    'Bulawayo': 'Bulawayo',
    'Manicaland': 'Manicaland',
    'Masvingo': 'Masvingo',
    'Midlands': 'Midlands',
    'Mashonaland Central': 'Mashonaland Central',
    'Mashonaland East': 'Mashonaland East',
    'Mashonaland West': 'Mashonaland West',
    'Matabeleland North': 'Matabeleland North',
    'Matabeleland South': 'Matabeleland South',
    'Chitungwiza': 'Harare',
    'Norton': 'Harare',
    'Bindura': 'Mashonaland Central',
    'Gweru': 'Midlands',
    'Kadoma': 'Midlands'
}

# Create base dataframe
years = np.arange(2010, 2024)
data_list = []

for district in districts_list:
    for year in years:
        data_list.append({
            'district': district,
            'province': provinces.get(district, 'Unknown'),
            'year': year,
        })

zimbabwe_data = pd.DataFrame(data_list)

# Generate realistic features
np.random.seed(42)

# SPEI Drought Index (realistic: increasing severity)
drought_trend = -0.02 * (zimbabwe_data['year'] - 2010)
drought_noise = np.random.normal(0, 0.3, len(zimbabwe_data))
zimbabwe_data['spei_drought'] = drought_trend + drought_noise
zimbabwe_data['spei_drought'] = zimbabwe_data['spei_drought'].clip(-3, 2)

# Nighttime Lights
lights_base = np.random.uniform(5, 30, len(zimbabwe_data))
lights_trend = 0.5 * (zimbabwe_data['year'] - 2010)
zimbabwe_data['night_lights'] = lights_base + lights_trend + np.random.normal(0, 2, len(zimbabwe_data))
zimbabwe_data['night_lights'] = zimbabwe_data['night_lights'].clip(0, 60)

# Agricultural Yield
ag_base = np.random.uniform(1000, 3000, len(zimbabwe_data))
ag_drought = zimbabwe_data['spei_drought'] * -150
ag_trend = -20 * (zimbabwe_data['year'] - 2010)
zimbabwe_data['ag_yield'] = ag_base + ag_drought + ag_trend + np.random.normal(0, 100, len(zimbabwe_data))
zimbabwe_data['ag_yield'] = zimbabwe_data['ag_yield'].clip(100, 4000)

# Education Index
edu_base = np.random.uniform(0.4, 0.75, len(zimbabwe_data))
edu_trend = 0.01 * (zimbabwe_data['year'] - 2010)
zimbabwe_data['education_index'] = edu_base + edu_trend + np.random.normal(0, 0.05, len(zimbabwe_data))
zimbabwe_data['education_index'] = zimbabwe_data['education_index'].clip(0, 1)

# Conflict Index
conflict_base = np.random.exponential(0.5, len(zimbabwe_data))
conflict_spikes = (zimbabwe_data['year'].isin([2016, 2019, 2020, 2023])).astype(int) * np.random.uniform(1, 3, len(zimbabwe_data))
zimbabwe_data['conflict_index'] = (conflict_base + conflict_spikes) * 10
zimbabwe_data['conflict_index'] = zimbabwe_data['conflict_index'].clip(0, 50)

# Migration Rate (dependent variable)
migration_base = 5 + np.random.normal(0, 1, len(zimbabwe_data))
migration_drought = -zimbabwe_data['spei_drought'] * 2
migration_lights = -zimbabwe_data['night_lights'] * 0.1
migration_ag = (3000 - zimbabwe_data['ag_yield']) * 0.005
migration_edu = (1 - zimbabwe_data['education_index']) * 5
migration_conflict = zimbabwe_data['conflict_index'] * 0.2

zimbabwe_data['migration_rate'] = (migration_base + migration_drought + migration_lights + 
                                    migration_ag + migration_edu + migration_conflict)
zimbabwe_data['migration_rate'] = zimbabwe_data['migration_rate'].clip(0, 30)

# Save CSV
csv_path = DATA_DIR / "zimbabwe_district_data.csv"
zimbabwe_data.to_csv(csv_path, index=False)

print(f"\n✓ Dataset generated: {len(zimbabwe_data)} records")
print(f"  Columns: {list(zimbabwe_data.columns)}")
print(f"  Districts: {zimbabwe_data['district'].nunique()}")
print(f"  Years: {zimbabwe_data['year'].min()}-{zimbabwe_data['year'].max()}")
print(f"\n{zimbabwe_data.head(10)}")

## Cell 3: Create Shapefile

In [ ]:
# Create mock shapefile with realistic district coordinates
district_coords = {
    'Harare': (31.05, -17.82),
    'Bulawayo': (28.58, -20.15),
    'Manicaland': (32.67, -18.69),
    'Masvingo': (30.27, -20.07),
    'Midlands': (29.27, -19.37),
    'Mashonaland Central': (30.77, -17.47),
    'Mashonaland East': (32.57, -17.95),
    'Mashonaland West': (29.77, -17.27),
    'Matabeleland North': (26.87, -18.57),
    'Matabeleland South': (27.37, -21.07),
    'Chitungwiza': (31.08, -18.02),
    'Norton': (30.77, -17.95),
    'Bindura': (30.27, -17.33),
    'Gweru': (29.27, -19.45),
    'Kadoma': (29.93, -18.33)
}

gdf = gpd.GeoDataFrame({
    'district': list(district_coords.keys()),
    'geometry': gpd.points_from_xy(
        [district_coords[d][0] for d in district_coords.keys()],
        [district_coords[d][1] for d in district_coords.keys()]
    )
})

shp_output = DATA_DIR / "zimbabwe_districts.shp"
gdf.to_file(shp_output, index=False)

print(f"✓ Shapefile created: {shp_output}")
print(f"  Districts: {len(gdf)}")

## Cell 4: Build & Train Bayesian Model

In [ ]:
import pymc as pm
import arviz as az
from sklearn.preprocessing import StandardScaler

print("="*70)
print("BUILDING BAYESIAN HIERARCHICAL CAR MODEL")
print("="*70)

# Reload data
data = pd.read_csv("/content/data/zimbabwe_district_data.csv")
gdf = gpd.read_file("/content/data/zimbabwe_districts.shp")

# Prepare indices
data = data.sort_values(['province', 'district', 'year'])
data['district_idx'] = data['district'].astype('category').cat.codes
data['province_idx'] = data['province'].astype('category').cat.codes
data['year_idx'] = data['year'] - data['year'].min()

n_districts = data['district'].nunique()

# Create adjacency matrix (simplified)
W = np.eye(n_districts)

# Features
features = ['spei_drought', 'night_lights', 'ag_yield', 'education_index', 'conflict_index']
X = data[features].values
scaler = StandardScaler()
X = scaler.fit_transform(X)
y = data['migration_rate'].values

print(f"Districts: {n_districts}")
print(f"Provinces: {data['province'].nunique()}")
print(f"Years: {data['year'].nunique()}")
print(f"Total observations: {len(data)}")

# Build model
with pm.Model() as model:
    # Global effects
    mu_global = pm.Normal("mu_global", 0, 2)
    sigma = pm.HalfNormal("sigma", 2)
    
    # Hierarchical Province Effect
    sigma_prov = pm.HalfNormal("sigma_prov", 1)
    province_effect = pm.Normal("province_effect", 0, sigma_prov, 
                              shape=data['province'].nunique())
    
    # Spatial Effect (simplified CAR)
    tau_phi = pm.Gamma("tau_phi", 1, 1)
    phi = pm.Normal("phi", 0, tau=tau_phi, shape=n_districts)
    
    # Temporal Effect
    sigma_time = pm.HalfNormal("sigma_time", 1)
    time_effect = pm.RandomWalk("time_effect", sigma=sigma_time, 
                              shape=data['year_idx'].max() + 1)
    
    # Fixed Effects
    beta = pm.Normal("beta", 0, 1, shape=len(features))
    
    # Linear Predictor
    mu = (mu_global +
          province_effect[data['province_idx'].values] +
          phi[data['district_idx'].values] +
          time_effect[data['year_idx'].values] +
          pm.math.dot(X, beta))
    
    # Likelihood
    obs = pm.StudentT("obs", nu=4, mu=mu, sigma=sigma, observed=y)

    # Sample
    print("\nSampling posterior (2000 draws, 1500 tuning)...")
    trace = pm.sample(draws=2000, tune=1500, target_accept=0.93, 
                     chains=2, cores=2, return_inferencedata=True, 
                     progressbar=True, random_seed=42)

# Save trace
trace_path = "/content/data/zimbabwe_migration_trace.nc"
az.to_netcdf(trace, trace_path)

print(f"\n✓ Model training complete!")
print(f"  Trace saved to: {trace_path}")

## Cell 5: Model Diagnostics & Summary

In [ ]:
import matplotlib.pyplot as plt

print("="*70)
print("BAYESIAN MODEL - POSTERIOR SUMMARY")
print("="*70)

summary = az.summary(trace, var_names=["beta", "mu_global", "sigma", "sigma_prov"])
print(summary)

# Posterior predictive checks
print("\n" + "="*70)
print("POSTERIOR PREDICTIVE CHECKS")
print("="*70)

with model:
    ppc = pm.sample_posterior_predictive(trace, random_seed=42)

az.plot_ppc(ppc, num_pp_samples=100)
plt.title("Posterior Predictive Check: Observed vs Simulated")
plt.tight_layout()
plt.savefig('/content/posterior_predictive_check.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Posterior predictive check plot saved")

## Cell 6: Data Exploration & Visualization

In [ ]:
# Reload data
data = pd.read_csv("/content/data/zimbabwe_district_data.csv")
trace = az.from_netcdf("/content/data/zimbabwe_migration_trace.nc")

print("="*70)
print("DATA SUMMARY")
print("="*70)

print(f"\nDataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
print(f"Districts: {data['district'].nunique()}")
print(f"Years: {data['year'].min()}-{data['year'].max()}")
print(f"Provinces: {data['province'].nunique()}")

print("\n" + "="*70)
print("DESCRIPTIVE STATISTICS")
print("="*70)

desc_stats = data[['migration_rate', 'spei_drought', 'night_lights', 'ag_yield', 'education_index', 'conflict_index']].describe()
print(desc_stats)

print("\n" + "="*70)
print("TOP 10 DISTRICTS BY AVERAGE MIGRATION RATE")
print("="*70)

top_migration = data.groupby('district')['migration_rate'].mean().sort_values(ascending=False).head(10)
print(top_migration)

print("\n" + "="*70)
print("CORRELATION MATRIX")
print("="*70)

corr = data[['migration_rate', 'spei_drought', 'night_lights', 'ag_yield', 'education_index', 'conflict_index']].corr()
print(corr)

## Cell 7: Visualization - Relationships

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Covariate Relationships with Migration Rate', fontsize=16, fontweight='bold')

# Migration Rate Distribution
axes[0, 0].hist(data['migration_rate'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Migration Rate Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Migration Rate (%)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(data['migration_rate'].mean(), color='red', linestyle='--', label=f"Mean: {data['migration_rate'].mean():.2f}")
axes[0, 0].legend()

# Drought vs Migration
axes[0, 1].scatter(data['spei_drought'], data['migration_rate'], alpha=0.5, s=20, color='orange')
z = np.polyfit(data['spei_drought'], data['migration_rate'], 1)
p = np.poly1d(z)
axes[0, 1].plot(data['spei_drought'].sort_values(), p(data['spei_drought'].sort_values()), "r--", alpha=0.8, linewidth=2)
corr_drought = data['spei_drought'].corr(data['migration_rate'])
axes[0, 1].set_title(f'Drought vs Migration (r={corr_drought:.3f})', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('SPEI Drought Index')
axes[0, 1].set_ylabel('Migration Rate (%)')

# Nighttime Lights vs Migration
axes[0, 2].scatter(data['night_lights'], data['migration_rate'], alpha=0.5, s=20, color='green')
z = np.polyfit(data['night_lights'], data['migration_rate'], 1)
p = np.poly1d(z)
axes[0, 2].plot(data['night_lights'].sort_values(), p(data['night_lights'].sort_values()), "r--", alpha=0.8, linewidth=2)
corr_lights = data['night_lights'].corr(data['migration_rate'])
axes[0, 2].set_title(f'Economic Activity vs Migration (r={corr_lights:.3f})', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Night Lights (NOAA VIIRS)')
axes[0, 2].set_ylabel('Migration Rate (%)')

# Ag Yield vs Migration
axes[1, 0].scatter(data['ag_yield'], data['migration_rate'], alpha=0.5, s=20, color='purple')
z = np.polyfit(data['ag_yield'], data['migration_rate'], 1)
p = np.poly1d(z)
axes[1, 0].plot(data['ag_yield'].sort_values(), p(data['ag_yield'].sort_values()), "r--", alpha=0.8, linewidth=2)
corr_ag = data['ag_yield'].corr(data['migration_rate'])
axes[1, 0].set_title(f'Agricultural Yield vs Migration (r={corr_ag:.3f})', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Ag Yield (kg/ha)')
axes[1, 0].set_ylabel('Migration Rate (%)')

# Education vs Migration
axes[1, 1].scatter(data['education_index'], data['migration_rate'], alpha=0.5, s=20, color='brown')
z = np.polyfit(data['education_index'], data['migration_rate'], 1)
p = np.poly1d(z)
axes[1, 1].plot(data['education_index'].sort_values(), p(data['education_index'].sort_values()), "r--", alpha=0.8, linewidth=2)
corr_edu = data['education_index'].corr(data['migration_rate'])
axes[1, 1].set_title(f'Education Index vs Migration (r={corr_edu:.3f})', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Education Index')
axes[1, 1].set_ylabel('Migration Rate (%)')

# Conflict vs Migration
axes[1, 2].scatter(data['conflict_index'], data['migration_rate'], alpha=0.5, s=20, color='red')
z = np.polyfit(data['conflict_index'], data['migration_rate'], 1)
p = np.poly1d(z)
axes[1, 2].plot(data['conflict_index'].sort_values(), p(data['conflict_index'].sort_values()), "r--", alpha=0.8, linewidth=2)
corr_conflict = data['conflict_index'].corr(data['migration_rate'])
axes[1, 2].set_title(f'Conflict Index vs Migration (r={corr_conflict:.3f})', fontsize=12, fontweight='bold')
axes[1, 2].set_xlabel('Conflict Events')
axes[1, 2].set_ylabel('Migration Rate (%)')

plt.tight_layout()
plt.savefig('/content/covariate_relationships.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: covariate_relationships.png")

## Cell 8: Model Interpretation

In [ ]:
print("="*70)
print("BAYESIAN MODEL INTERPRETATION")
print("="*70)

# Extract beta coefficients
beta_summary = az.summary(trace, var_names=["beta"])
print("\nFEATURE EFFECTS (Beta Coefficients):")
print("Positive = increases migration | Negative = decreases migration")
print(beta_summary)

# Get mean values
features = ['spei_drought', 'night_lights', 'ag_yield', 'education_index', 'conflict_index']
beta_values = trace.posterior["beta"].mean(dim=["chain", "draw"]).values

print("\n" + "="*70)
print("KEY FINDINGS: Which factors drive migration most?")
print("="*70)

sorted_effects = sorted(zip(features, beta_values), key=lambda x: abs(x[1]), reverse=True)

for i, (feature, effect) in enumerate(sorted_effects, 1):
    direction = "↑ INCREASES" if effect > 0 else "↓ DECREASES"
    magnitude = abs(effect)
    emoji = "🔴" if magnitude > 0.5 else "🟡" if magnitude > 0.2 else "🟢"
    print(f"\n{i}. {feature.upper().replace('_', ' ')}")
    print(f"   {emoji} Effect: {direction} migration")
    print(f"   Magnitude: {magnitude:.3f} (per standard deviation increase)")

# Global statistics
print("\n" + "="*70)
print("GLOBAL MODEL STATISTICS")
print("="*70)

mu_global = trace.posterior["mu_global"].mean().values
sigma = trace.posterior["sigma"].mean().values

print(f"\nGlobal Mean Migration Rate: {mu_global:.3f}%")
print(f"Unexplained Variance (sigma): {sigma:.3f}%")
print(f"\nModel captures ~{100 - (sigma/data['migration_rate'].std())*100:.1f}% of variance")

## Cell 9: Posterior Trace Plots

In [ ]:
# Trace plots for convergence diagnostics
az.plot_trace(trace, var_names=['beta', 'mu_global', 'sigma'], figsize=(12, 8))
plt.tight_layout()
plt.savefig('/content/posterior_trace_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Trace plots saved: posterior_trace_plots.png")
print("\nTrace plots show:")
print("- LEFT column: Posterior distributions (should look smooth/bell-shaped)")
print("- RIGHT column: MCMC chains (should look like white noise, not trends)")

## Cell 10: Summary & Next Steps

In [ ]:
print("\n" + "="*70)
print("✓ COMPLETE ANALYSIS PIPELINE FINISHED")
print("="*70)

print("\n📊 OUTPUTS GENERATED:")
print("  1. zimbabwe_district_data.csv - Dataset (1,820 records)")
print("  2. zimbabwe_districts.shp - Spatial boundaries")
print("  3. zimbabwe_migration_trace.nc - MCMC samples")
print("  4. posterior_predictive_check.png - Model diagnostics")
print("  5. covariate_relationships.png - Scatter plots with correlations")
print("  6. posterior_trace_plots.png - Convergence diagnostics")

print("\n📥 DOWNLOAD YOUR FILES:")
print("  Files are in /content/data/ and /content/*.png")
print("  Use the file browser on the left to download them")

print("\n🚀 NEXT STEPS:")
print("  1. Download the PNG visualizations to your computer")
print("  2. Use the CSV data for further analysis")
print("  3. Review model diagnostics (trace plots, posterior checks)")
print("  4. Run counterfactual policy simulations (edit counterfactuals.py)")
print("  5. Deploy interactive dashboard (streamlit run app.py)")

print("\n" + "="*70)